# 01 — Explore Subsidence Results
**Urban Subsidence Watcher** | Samuel Appiah Kubi | PLUS Salzburg

This notebook loads the processed subsidence rate map and associated products, 
visualises the spatial distribution of subsidence across Accra, Ghana, and 
generates summary statistics for reporting.

**Pre-requisites:** Run `python main.py --step rates` before opening this notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.plot import show
from pathlib import Path
import yaml
import warnings
warnings.filterwarnings('ignore')

# Load project config
with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

DATA = Path('../data')
print('Config loaded. Study area:', cfg['study_area']['name'])

## 1. Load Subsidence Rate Map

In [ ]:
rate_path = DATA / 'subsidence' / 'rate_map.tif'
error_path = DATA / 'subsidence' / 'rate_error.tif'
seasonal_path = DATA / 'subsidence' / 'seasonal_amplitude.tif'

if not rate_path.exists():
    print('[WARNING] rate_map.tif not found — generating synthetic data for demonstration.')
    rng = np.random.default_rng(42)
    rate = rng.uniform(-30, 2, (200, 200)).astype(np.float32)
    error = rng.uniform(0, 3, (200, 200)).astype(np.float32)
    seasonal = np.abs(rng.normal(3, 1, (200, 200))).astype(np.float32)
    meta = None
else:
    with rasterio.open(rate_path) as src:
        rate = src.read(1)
        meta = src.meta
    with rasterio.open(error_path) as src:
        error = src.read(1)
    with rasterio.open(seasonal_path) as src:
        seasonal = src.read(1)

print(f'Rate map shape: {rate.shape}')
print(f'Rate range: {rate.min():.1f} to {rate.max():.1f} mm/yr')
print(f'Mean rate: {rate.mean():.2f} mm/yr')
print(f'Pixels with rate < -5 mm/yr: {(rate < -5).sum():,}')

## 2. Spatial Distribution of Subsidence

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Urban Subsidence Watcher — Accra, Ghana', fontsize=14, fontweight='bold')

# Rate map
cmap_rate = plt.cm.RdYlGn
vmin, vmax = -30, 5
im0 = axes[0].imshow(rate, cmap=cmap_rate, vmin=vmin, vmax=vmax)
axes[0].set_title('Subsidence Rate (mm/yr)')
plt.colorbar(im0, ax=axes[0], label='mm/yr')

# Error map
im1 = axes[1].imshow(error, cmap='hot_r', vmin=0, vmax=5)
axes[1].set_title('Rate Error (mm/yr)')
plt.colorbar(im1, ax=axes[1], label='mm/yr')

# Seasonal amplitude
im2 = axes[2].imshow(seasonal, cmap='Blues', vmin=0, vmax=10)
axes[2].set_title('Seasonal Amplitude (mm)')
plt.colorbar(im2, ax=axes[2], label='mm')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(DATA / 'validation' / 'subsidence_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 3. Histogram of Subsidence Rates

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
valid = rate[np.isfinite(rate) & (rate > -100) & (rate < 20)]
ax.hist(valid.ravel(), bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(-5, color='orange', linestyle='--', linewidth=1.5, label='Hotspot threshold (−5 mm/yr)')
ax.axvline(valid.mean(), color='red', linestyle='-', linewidth=1.5, label=f'Mean = {valid.mean():.1f} mm/yr')
ax.set_xlabel('Subsidence Rate (mm/yr)', fontsize=12)
ax.set_ylabel('Pixel Count', fontsize=12)
ax.set_title('Distribution of Annual Subsidence Rates — Accra', fontsize=13)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Summary Statistics

In [ ]:
stats = {
    'Metric': [
        'Mean rate (mm/yr)', 'Median rate (mm/yr)',
        'Max subsidence (mm/yr)', 'Std deviation (mm/yr)',
        'Hotspot pixels (< −5 mm/yr)', 'Hotspot fraction (%)',
        'Mean seasonal amplitude (mm)', 'Mean rate error (mm/yr)',
    ],
    'Value': [
        f'{valid.mean():.2f}',
        f'{np.median(valid):.2f}',
        f'{valid.min():.2f}',
        f'{valid.std():.2f}',
        f'{(valid < -5).sum():,}',
        f'{100 * (valid < -5).mean():.1f}',
        f'{seasonal[np.isfinite(seasonal)].mean():.2f}',
        f'{error[np.isfinite(error)].mean():.2f}',
    ]
}

df_stats = pd.DataFrame(stats)
df_stats.style.set_caption('Subsidence Statistics — Accra, Ghana')

## 5. Time-Series at Selected Points

In [ ]:
ts_path = DATA / 'subsidence' / 'timeseries.parquet'

if ts_path.exists():
    df_ts = pd.read_parquet(ts_path)
    print(f'Time series loaded: {df_ts.shape[0]} epochs, {df_ts.shape[1]-1} pixels')
    
    # Plot first 5 pixels
    fig, ax = plt.subplots(figsize=(11, 4))
    pixel_cols = [c for c in df_ts.columns if c != 'date'][:5]
    for col in pixel_cols:
        ax.plot(pd.to_datetime(df_ts['date']), df_ts[col], marker='o', ms=3, label=col)
    ax.set_xlabel('Date')
    ax.set_ylabel('Displacement (mm)')
    ax.set_title('Displacement Time Series — Selected Pixels')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('[INFO] timeseries.parquet not found. Run subsidence_rates.py first.')
    
    # Synthetic demo
    t = pd.date_range('2020-01-01', periods=50, freq='12D')
    fig, ax = plt.subplots(figsize=(11, 4))
    for rate_val in [-5, -12, -20, -2, -8]:
        ts = rate_val/365 * np.arange(50) + np.random.normal(0, 1, 50).cumsum() * 0.2
        ax.plot(t, ts, marker='o', ms=2, label=f'{rate_val} mm/yr (synthetic)')
    ax.set_xlabel('Date')
    ax.set_ylabel('Displacement (mm)')
    ax.set_title('Displacement Time Series — Synthetic Demo')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Forecast Maps

In [ ]:
forecast_dir = DATA / 'forecasts'
horizons = [1, 3, 5]
forecasts = []

for h in horizons:
    p = forecast_dir / f'forecast_{h}yr.tif'
    if p.exists():
        with rasterio.open(p) as src:
            forecasts.append(src.read(1))
    else:
        forecasts.append(rate * h)  # synthetic fallback

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Subsidence Forecasts — Accra, Ghana', fontsize=14, fontweight='bold')

for ax, f, h in zip(axes, forecasts, horizons):
    im = ax.imshow(f, cmap='RdYlGn', vmin=-150, vmax=10)
    ax.set_title(f'{h}-Year Forecast (mm)')
    plt.colorbar(im, ax=ax, label='mm')
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Forecast visualisation complete.')